# EXP_040B — Fusion Ablation: GMU + MSE
**Phase 4 | Fusion Upgrade**
Research question: Does Gated Multimodal Unit improve over simple Concatenation?
- Fusion: `gmu` | Text: Best from Phase 3 | Image: Best from Phase 2 | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: Phase 2 and Phase 3 must be completed. Set BEST_IMAGE_EXP_ID and BEST_TEXT_MODEL in STEP 4.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13379, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 13379 (delta 185), reused 165 (delta 103), pack-reused 13129 (from 1)
Receiving objects: 100% (13379/13379), 873.22 MiB | 19.94 MiB/s, done.
Resolving deltas: 100% (423/423), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1344
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 23 16:18 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths — ✏️ Set Phase 2 & Phase 3 winners here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_040B_bestimage_besttext_gmu_mse'

# ✏️ SET based on Phase 2 results (lowest Mean MAE wins)
# Options: 'EXP_020B_swinb_xlmr_concat_mse' / 'EXP_020D_...' / 'EXP_020E_...'
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'

# ✏️ SET based on Phase 3 results (lowest Mean MAE wins)
# Options: 'EXP_030B_bestimage_phobert_concat_mse' / 'EXP_030D_...'
# If XLM-R still wins Phase 3, set: BEST_TEXT_EXP_ID = 'EXP_010_text_only_xlmr_mse', BEST_TEXT_MODEL = 'xlm-roberta-base'
BEST_TEXT_EXP_ID = 'EXP_030B_bestimage_phobert_concat_mse'
BEST_TEXT_MODEL  = 'vinai/phobert-base-v2'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')
print(f'Image backbone : {BEST_IMAGE_MODEL}')
print(f'Text backbone  : {BEST_TEXT_MODEL}')

Artifacts will be saved to: /content/drive/MyDrive/SE365/experiments/EXP_040B_bestimage_besttext_gmu_mse
Image backbone : swin_base_patch4_window7_224
Text backbone  : vinai/phobert-base-v2


### STEP 5: Load pretrained weights from Phase 2 & Phase 3 winners

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text weights from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image weights from {BEST_IMAGE_EXP_ID}')

Loaded text weights from EXP_030B_bestimage_phobert_concat_mse
Loaded image weights from EXP_020B_swinb_xlmr_concat_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type gmu \
  --text_model_name {BEST_TEXT_MODEL} \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_040B_bestimage_besttext_gmu_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_040B_bestimage_besttext_gmu_mse
config.json: 100% 678/678 [00:00<00:00, 3.05MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 100MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 151MB/s]
Loaded timm processor for swin_base_patch4_window7_224
pytorch_model.bin: 100% 540M/540M [00:04<00:00, 132MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 23248.58it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different ta

### STEP 7: Save to Drive + print metrics

In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")


=== EXP_040B_bestimage_besttext_gmu_mse Results ===
Loss (val)   : 2.2047

             MAE      RMSE      R2
  food     : 1.1136   1.5068   0.5686
  price    : 1.1756   1.5698   0.4483
  atmos    : 1.1813   1.5245   0.4012
  service  : 1.1808   1.5671   0.5210
  overall  : 0.9289   1.2364   0.6246

  mean_mae   : 1.1160
  aspect_mae : 1.1628
  overall_mae: 0.9289
